### Manual: 12-checkpoint-load

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


In [ ]:
"""Load the official fused checkpoint and list every key mismatch.

`step_checkpoint.py` builds the model, loads the checkpoint with
`load_state_dict(strict=False)` (a shape mismatch still raises, so it is caught
and reported), cross-checks with `mmcv.runner.load_checkpoint`, and verifies a
sample of weights really landed in the module.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

rc, text = sc.run_script("step_checkpoint.py", log_name="step_checkpoint.log", timeout=2400)
payload = sc.load_json("checkpoint.json")
ckpt = payload.get("checkpoint", {})
print()
print("rc = %d" % rc)
print("sha256        : %s" % ckpt.get("sha256"))
print("size_bytes    : %s" % ckpt.get("size_bytes"))
print("ckpt keys     : %s" % ckpt.get("ckpt_num_keys"))
print("model keys    : %s" % ckpt.get("model_num_keys"))
print("matched keys  : %s" % ckpt.get("matched_keys"))
print("shape mismatch: %s" % ckpt.get("shape_mismatch_count"))
print("missing keys  : %s (weight affecting: %s)"
      % (ckpt.get("missing_keys_total"), ckpt.get("missing_keys_weight_affecting_total")))
for key in ckpt.get("missing_keys", [])[:40]:
    print("   MISSING    " + key)
print("unexpected    : %s" % ckpt.get("unexpected_keys_total"))
for key in ckpt.get("unexpected_keys", [])[:40]:
    print("   UNEXPECTED " + key)
print("load_error    : %s" % ckpt.get("load_error"))
print("mmcv cross    : %s" % json.dumps(ckpt.get("cross_check_mmcv")))
print("weight sample : %s" % json.dumps(ckpt.get("weight_sample_check")))
print("CKPT_CELL_OK" if ckpt.get("ok") else "CKPT_CELL_FAILED (see logs/step_checkpoint.log)")
